# 26.9.14

In [1]:
import torch
print(torch.__version__)              # 2.7.1+cu128
print(torch.cuda.is_available())      # True 나와야 GPU
print(torch.cuda.get_device_name(0))  # RTX 5060

2.7.1+cu128
True
NVIDIA GeForce RTX 5060 Laptop GPU


In [2]:
print(torch.tensor([1,2,3]))              # 1차원 (OK)
print(torch.Tensor([[1,2,3],[4,5,6]]))    # 2차원 (감쌈)
print(torch.LongTensor([1,2,3]))          # OK
print(torch.FloatTensor([1,2,3]))  

tensor([1, 2, 3])
tensor([[1., 2., 3.],
        [4., 5., 6.]])
tensor([1, 2, 3])
tensor([1., 2., 3.])


In [3]:
tensor = torch.rand(1,2)
print(tensor.shape)
print(tensor.dtype)
print(tensor.device)

torch.Size([1, 2])
torch.float32
cpu


In [4]:
tensor = tensor.reshape(2, 1)
print(tensor)
print(tensor.shape)

tensor([[0.7461],
        [0.4348]])
torch.Size([2, 1])


In [5]:
tensor = torch.rand((3,3), dtype = torch.float)
print(tensor)

tensor([[0.9264, 0.4995, 0.0077],
        [0.1141, 0.7156, 0.4332],
        [0.7103, 0.0799, 0.8419]])


In [6]:
device = "cuda" if torch.cuda.is_available() else 'cpu'   # cuda 추가 + 철자
print(device)

cuda


In [7]:
cpu = torch.FloatTensor([1, 2, 3])
gpu = torch.cuda.FloatTensor([1, 2, 3])
tensor = torch.rand((1, 1), device = device)
print(cpu)
print(gpu)
print(tensor)

C:\Users\wm032\AppData\Local\Temp\ipykernel_41048\4063380246.py:2: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\tensor\python_tensor.cpp:80.)
  gpu = torch.cuda.FloatTensor([1, 2, 3])


tensor([1., 2., 3.])
tensor([1., 2., 3.], device='cuda:0')
tensor([[0.2246]], device='cuda:0')


In [8]:
cpu = torch.FloatTensor([1, 2, 3])
gpu = cpu.cuda()
gpu2cpu = gpu.cpu()
cpu2gpu = cpu.to(device)

In [9]:
import numpy as np

ndarray = np.array([1, 2, 3], dtype = np.uint8)
print(torch.tensor(ndarray))
print(torch.Tensor(ndarray))
print(torch.from_numpy(ndarray))

tensor([1, 2, 3], dtype=torch.uint8)
tensor([1., 2., 3.])
tensor([1, 2, 3], dtype=torch.uint8)


In [10]:
tensor = torch.cuda.FloatTensor([1,2,3])
ndarray = tensor.detach().cpu().numpy()
print(ndarray)
print(type(ndarray))

[1. 2. 3.]
<class 'numpy.ndarray'>


# 비선형회귀 만들기(torch)

In [11]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from  torch.utils.data import Dataset, DataLoader, random_split
from pathlib import Path

# 노트북 파일이 있는 폴더 기준 (실행 위치와 상관없이 동일)
NOTEBOOK = globals().get('__vsc_ipynb_file__')
BASE_DIR = Path(NOTEBOOK).parent if NOTEBOOK else Path.cwd()
MODEL_DIR = BASE_DIR / 'models'
MODEL_DIR.mkdir(exist_ok=True)

In [12]:
class CustomDataset(Dataset):
    def __init__(self, file_path):
        df = pd.read_csv(file_path)
        self.x = df.iloc[:, 0].values
        self.y = df.iloc[:,1].values
        self.length = len(df)
    
    def __getitem__(self,index):
        x = torch.FloatTensor([self.x[index] ** 2, self.x[index]])
        y = torch.FloatTensor([self.y[index]])
        return x, y
    
    def __len__(self):
        return self.length

In [13]:
class CustomModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(2, 1)

    def forward(self, x):
        x = self.layer(x)
        return x

In [14]:
dataset = CustomDataset(BASE_DIR / 'non_linear.csv')
dataset_size = len(dataset)
train_size = int(dataset_size * 0.8)
val_size = int(dataset_size * 0.1)
test_size = dataset_size - train_size - val_size 

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

train_dataloader = DataLoader(
    train_dataset, 
    batch_size = 16, 
    shuffle = True, 
    drop_last = True
    )

val_dataloader = DataLoader(
    val_dataset, 
    batch_size = 4, 
    shuffle = True, 
    drop_last = True
    )

test_dataloader = DataLoader(
    test_dataset, 
    batch_size = 4, 
    shuffle = False, 
    drop_last = True
    )    

In [15]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = CustomModel().to(device)
criterion = nn.MSELoss().to(device)
optimizer = optim.SGD(model.parameters(), lr = 0.0001)

In [16]:
checkpoint = 1
for epoch in range(1000):
    cost = 0.0
    for x, y in train_dataloader:
        x = x.to(device)
        y = y.to(device)
        output = model(x)
        loss = criterion(output, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        cost += loss.item()
    cost = cost / len(train_dataloader)
    if (epoch + 1) % 100 == 0:
        print(epoch + 1, cost)
        torch.save(model.state_dict(), MODEL_DIR / f'checkpoint-{checkpoint}.pt')
        checkpoint += 1

100 0.2623759612441063
200 0.23034038692712783
300 0.20333704054355622
400 0.1851567454636097
500 0.16671877428889276
600 0.15159695781767368
700 0.13837983533740045
800 0.12943755760788916
900 0.12415166720747947
1000 0.11394343599677086


In [17]:
with torch.no_grad():
    model.eval()
    for x, y in val_dataloader:
        x = x.to(device)
        y = y.to(device)
        outputs = model(x)
        print(outputs)


tensor([[  9.2353],
        [228.2061],
        [116.4777],
        [ 67.8125]], device='cuda:0')
tensor([[  1.2054],
        [114.5248],
        [120.3120],
        [ 23.8810]], device='cuda:0')
tensor([[171.1655],
        [ 22.1897],
        [ 59.3859],
        [255.5987]], device='cuda:0')
tensor([[18.2565],
        [32.2917],
        [ 0.5526],
        [28.4104]], device='cuda:0')
tensor([[ 41.8969],
        [217.6836],
        [ 31.2672],
        [212.5155]], device='cuda:0')


In [18]:
torch.save(model.state_dict(), MODEL_DIR / 'model.pt')

In [19]:
torch.save(model.state_dict(), MODEL_DIR / 'model_state_dict.pt')

In [20]:
model = CustomModel().to(device)
model.load_state_dict(torch.load(MODEL_DIR / 'model.pt', map_location = device))

<All keys matched successfully>

In [21]:
print(model)

CustomModel(
  (layer): Linear(in_features=2, out_features=1, bias=True)
)


In [22]:
model = CustomModel().to(device)
model_state_dict = torch.load(MODEL_DIR / 'model_state_dict.pt', map_location = device)
model.load_state_dict(model_state_dict)

<All keys matched successfully>

In [23]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from pathlib import Path

NOTEBOOK = globals().get('__vsc_ipynb_file__')
BASE_DIR = Path(NOTEBOOK).parent if NOTEBOOK else Path.cwd()
MODEL_DIR = BASE_DIR / 'models'
MODEL_DIR.mkdir(exist_ok=True)

In [24]:
class CustomDataset(Dataset):
    def __init__(self, file_path):
        df = pd.read_csv(file_path)
        self.x1 = df.iloc[:, 0].values
        self.x2 = df.iloc[:, 1].values
        self.x3 = df.iloc[:, 2].values
        self.y = df.iloc[:, 3].values
        self.length = len(df)

    def __getitem__(self, index):
        x = torch.FloatTensor([self.x1[index], self.x2[index], self.x3[index]])
        y = torch.FloatTensor([int(self.y[index])])
        return x, y
    
    def __len__(self):
        return self.length

In [25]:
class CustomModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Linear(3, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.layer(x)
        return x

In [26]:
dataset = CustomDataset(BASE_DIR / 'binary.csv')
dataset_size = len(dataset)
train_size = int(dataset_size * 0.8)
val_size = int(dataset_size * 0.1)
test_size = dataset_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset, [train_size, val_size, test_size], generator = torch.Generator().manual_seed(42)
    )

train_dataloader = DataLoader(
    train_dataset, 
    batch_size = 64, 
    shuffle = True, 
    drop_last = True
    )

val_dataloader = DataLoader(
    val_dataset, 
    batch_size = 4, 
    shuffle = True, 
    drop_last = True
    )

test_dataloader = DataLoader(
    test_dataset, 
    batch_size = 4, 
    shuffle = False, 
    drop_last = True
    )    

# 가중치 초기화함수